In [ ]:
import os
import geopandas as gpd
from shapely.geometry import Polygon
from geojson import Feature, FeatureCollection
from time import sleep
from selenium import webdriver
from selenium.webdriver.chrome.service import Service
from selenium.webdriver.common.by import By  # Import the By class
from selenium.webdriver.support.ui import WebDriverWait
from selenium.webdriver.support import expected_conditions as EC
from selenium.webdriver.chrome.options import Options
from bs4 import BeautifulSoup
from splinter import Browser
import pandas as pd
import zipfile

In [ ]:
from modules.functions import create_query_url

In [ ]:
df = pd.read_excel("./background_files/אומדן יחד שכונות מעירית בית שמש יולי 2020.xlsx")

In [ ]:
plans_numbers = df['תב"ע']

In [ ]:
plans_numbers.dropna(inplace=True)

In [ ]:
xplan_plans_data = []

In [ ]:
options = Options()

# options.add_argument("--headless=new")

driver = webdriver.Chrome(options=options)

driver.get("https://mavat.iplan.gov.il/SV1")

# driver.quit()

In [ ]:
# html = driver.page_source

# soup = BeautifulSoup(html, 'html.parser')

# # מציאת כל האלמנטים עם המחלקה uk-margin-remove
# pl_name_element = soup.find_all(class_="sv4-icon-flag ng-star-inserted")
# # חיפוש בתוך ResultSet אחר טקסט מסוים
# for element in pl_name_element:
#     a = element.find('div', class_="uk-accordion-content uk-margin-remove")

#     b = a.find('div', class_="uk-grid uk-grid-collapse sv4-data default-background ng-star-inserted")

#     c = b.find('div', class_="uk-width-2-3")

#     d = c.find('div', class_="uk-grid uk-grid-collapse ng-star-inserted")

#     e = d.find('div', class_="uk-padding-small")
#     print(e.get_text(separator=' ', strip=True).replace("ישוב ", ""))

In [ ]:
plans_numbers = plans_numbers[plans_numbers['value'].isna()].index

for plan in plans_numbers:
    print(plan)
    plan_attributes = {
    'attributes': {
        'pl_number': "",               # מספר התוכנית (ריק בהתחלה)
        'pl_name': "",                 # שם התוכנית (ריק בהתחלה)
        'pl_url': "",                  # קישור לתוכנית (ריק בהתחלה)
        'station_desc': "",            # תיאור סטטוס (ריק בהתחלה)
        'plan_county_name': "",        # שם מחוז התוכנית (ריק בהתחלה)
        },
    }

    wait = WebDriverWait(driver, 10)
    search_box = wait.until(EC.visibility_of_element_located((By.ID, "sv3-search__input")))
    search_button = wait.until(EC.element_to_be_clickable((By.XPATH, "//button[@aria-label='חיפוש']")))
    search_box.send_keys(plan)
    
    search_button.click()

    sleep(5)

    print(driver.current_url)

    if 'SV4' not in driver.current_url:
        button = wait.until(EC.presence_of_element_located((By.XPATH, '//*[@id="pr_id_3-table"]/tbody/tr/td[10]/a')))
        print(button)
        button.click()

    sleep(10)

    # קבלת ה-HTML אחרי שהדף טוען את התוכן
    html = driver.page_source
    soup = BeautifulSoup(html, 'html.parser')

    stick = wait.until(EC.presence_of_element_located((By.XPATH, '//*[@id="sv4body"]/div[1]/div')))

    driver.execute_script("arguments[0].style.position = 'absolute';", stick)

    pl_number_element = soup.find_all(class_="uk-width-expand sv4-h1-content")

    if pl_number_element:
        for element in pl_number_element:
            plan_number = element.find('h1', class_="uk-margin-remove plan-name header-h1").text.split(' ')[1]
    else:
        print("pl_number: לא נמצא אלמנט עם שם המחלקה הזה!")

    pl_number_element = ''

    # מחפש את האלמנט <h2> עם מחלקה 'uk-margin-remove'
    pl_name_element = soup.find_all(class_="uk-width-expand sv4-h1-content")

    if pl_name_element:
        for element in pl_name_element:
            plan_attributes['attributes']['pl_name'] = element.find('h1', class_="uk-margin-remove plan-name header-h1").text
    else:
        print("pl_name: לא נמצא אלמנט עם המחלקה הזו!")

    pl_name_element = ''

    plan_attributes['attributes']['pl_url'] = driver.current_url

    pl_station_desc_element = soup.find_all(class_="uk-width-1-3@m sv4-h1-aside ng-star-inserted")

    if pl_station_desc_element:
        for element in pl_station_desc_element:
            plan_attributes['attributes']['station_desc'] = element.find(class_="h3 uk-margin-remove").text
    else:
        print("station_desc: לא נמצא אלמנט עם המחלקה הזו!")

    pl_station_desc_element = ''

    location_list_elements = soup.find_all(class_="sv4-icon-flag ng-star-inserted")

    if location_list_elements:
        for element in location_list_elements:
            a = element.find('div', class_="uk-accordion-content uk-margin-remove")

            b = a.find('div', class_="uk-grid uk-grid-collapse sv4-data default-background ng-star-inserted")

            c = b.find('div', class_="uk-width-2-3")

            d = c.find('div', class_="uk-grid uk-grid-collapse ng-star-inserted")

            e = d.find('div', class_="uk-padding-small")

            plan_attributes['attributes']['plan_county_name'] = e.get_text(separator=' ', strip=True).replace("ישוב ", "")
    else:
        print("plan_county_name: לא נמצא אלמנט עם המחלקה הזו!")

    location_list_elements = ''

    # הדפסת כותרות התוצאות או מידע אחר
    button = wait.until(EC.presence_of_element_located((By.XPATH, '//*[@id="moreQuantitiesFocus"]')))
    if button:
        button.click()

    lis_elements = []

    mydivs = soup.find_all("li", {"class": "sv4-icon-arrow uk-open uk-hide-arrow ng-star-inserted"})

    for divs in mydivs:
        div = divs.find_all('div', attrs={'class': 'uk-accordion-content uk-margin-remove'})

        results = div[0].find_all('ul')

    for result in results:
            li = result.find_all('li')
            lis_elements.append(li)
                
    obj = {}
    for element in lis_elements:
                quantitative_data_main_header = element[0].find_all('div', {'class': 'uk-width-expand ng-star-inserted'})[0].text.strip()
                quantitative_data_main_value = element[0].find_all('div', {'class': 'uk-width-1-2 uk-text-left'})[0].find_all('b')[0].text
                obj[quantitative_data_main_header] = quantitative_data_main_value

    plan_attributes['attributes'].update(obj)

    catA_element = wait.until(EC.visibility_of_element_located((By.XPATH, '//*[@id="sv4body"]/div[2]/div/div/div[1]/ul[2]/li[2]')))

    if catA_element:
        catA_element.click()
    else:
        print("האלמנט אינו נראה")

    catC_element = wait.until(EC.visibility_of_element_located((By.XPATH, '//*[@id="sv4body"]/div[2]/div/div/div[1]/ul[2]/li[2]/div[2]/div[2]/div/div/div[2]')))

    if catC_element:
        catC_element.click()
    else:
        print("האלמנט אינו נראה")

    download_elements = catC_element.find_elements(By.CLASS_NAME, "uk-grid.uk-grid-collapse.uk-flex.uk-flex-between.uk-flex-middle")

    for element in download_elements:
        if 'shp' in element.text or 'SHP' in element.text:
            print('shp')
            download_elements = element.find_elements(By.TAG_NAME, "a")
            download_elements[0].click()

    # download_elements = wait.until(EC.visibility_of_element_located((By.XPATH, '//*[@id="sv4body"]/div[2]/div/div/div[1]/ul[2]/li[2]/div[2]/div[2]/div/div/div[2]/ul/li/div[2]/div[5]/div/div/ul/li/div/div[4]/div/div/a')))
    # download_elements.click()

    sleep(10)

    # pr# נתיב לתיקיית ההורדות
    download_dir = r'C:\Users\dpere\Downloads'

    # ה-ID של הקובץ שאתה מחפש
    id_to_find = plan

    # רשום את כל הקבצים בתיקיית ההורדות
    files = os.listdir(download_dir)    

    # סנן את הקבצים לפי ה-ID שמופיע בשמם
    matching_files = [f for f in files if id_to_find in f]

    # הדפס את כל הקבצים שמתאימים ל-ID
    if matching_files:
        print('matching_files')
        zip_file_path = os.path.join(download_dir, matching_files[0])

        extract_to_folder = r'..\ags-iplan-data-fetcher-python\background_files'

        # יצירת תיקיית יעד עם שם ה-ID
        extract_to_folder = os.path.join(extract_to_folder, id_to_find)

        # אם תיקיית היעד לא קיימת, ניצור אותה
        os.makedirs(extract_to_folder, exist_ok=True)
    
        # לפתוח את קובץ ה-ZIP
        with zipfile.ZipFile(zip_file_path, 'r') as zip_ref:
            # רשימה של כל הקבצים ב-ZIP
            file_names = zip_ref.namelist()

            # סינון קבצים שמתחילים ב-'MVT_GVUL'
            mvt_gvul_files = [f for f in file_names if f.startswith('MVT_GVUL')]
            kavim_kchulim_files = [f for f in file_names if f.startswith('kavim_kchulim')]

            print(len(mvt_gvul_files) > 0)
            print(kavim_kchulim_files)

            if len(mvt_gvul_files) > 0:
                # פרוק כל הקבצים שמתחילים ב-'MVT_GVUL'
                for file_name in mvt_gvul_files:
                    zip_ref.extract(file_name, extract_to_folder)

            elif len(kavim_kchulim_files) > 0:
                # פרוק כל הקבצים שמתחילים ב-'kavim_kchulim'
                for file_name in kavim_kchulim_files:
                    zip_ref.extract(file_name, extract_to_folder)
    # # הגדרת תיקיית ההורדות ונתיב לתיקיה חדשה עם שם ה-ID
    # download_dir = r"C:\Users\dpere\Documents\JTMT\ags-iplan-data-fetcher-python\background_files"
    # id_to_find = plan  # ID שלך (שיש לקחת לפי המקרה שלך)

    # # יצירת תיקיה בשם ה-ID
        # folder_name = os.path.join(extract_to_folder, id_to_find)
    # # os.makedirs(folder_name, exist_ok=True)

        shp_file = None

        # הנתיב לקובץ ה-SHP (שכבה גיאוגרפית)
        if len(mvt_gvul_files) > 0:
            shp_file = os.path.join(extract_to_folder, 'MVT_GVUL.shp')
        elif len(kavim_kchulim_files) > 0:
            shp_file = os.path.join(extract_to_folder, 'kavim_kchulim.shp')

        # טעינת קובץ ה-SHP בעזרת GeoPandas
        gdf = gpd.read_file(shp_file)

        # כעת, נקבל את המידע הגיאומטרי - נגיד שהשכבה היא פוליגון (polygon)
        geometry = gdf.geometry.iloc[0]  # או לפי אינדקס מתאים

        # המרת הגיאומטריה למבנה של 'rings'
        rings = [list(geometry.exterior.coords)]  # מייצגים את הקואורדינטות כ-rings

        # יצירת האובייקט
        geo_object = {
                    "attributes": plan_attributes['attributes'],
                    "geometry": {
                        "rings": rings,
                    },
                }

        xplan_plans_data.append(geo_object)

        search_box = ''

        driver.get("https://mavat.iplan.gov.il/SV1")

In [ ]:
matching_files = [f for f in files if id_to_find in f]
matching_files

In [ ]:
PATH = 'chromedriver-win64/chromedriver.exe'

service = Service(executable_path=PATH)

# browser = Browser('chrome', headless=True)
browser = Browser('chrome')

# טוען את האתר
browser.visit('https://mavat.iplan.gov.il/SV1')

In [ ]:
lis_elements = []

for plan in plans_numbers:
    print(plan)
    plan_attributes = {
    'attributes': {
        'pl_number': "",               # מספר התוכנית (ריק בהתחלה)
        'pl_name': "",                 # שם התוכנית (ריק בהתחלה)
        'pl_url': "",                  # קישור לתוכנית (ריק בהתחלה)
        'station_desc': "",            # תיאור סטטוס (ריק בהתחלה)
        'plan_county_name': "",        # שם מחוז התוכנית (ריק בהתחלה)
    },
}
    # מחפש את שדה החיפוש לפי ID
    search_box = browser.find_by_id('sv3-search__input')

    if search_box:
        # ממלא את השדה במילת החיפוש
        search_box.fill('102-0074732')

        # שולח את החיפוש (לרוב עם Enter)
        search_box.first.type('\n')

        # המתנה לטעינת תוצאות
        sleep(7)

        pl_number_element = browser.find_by_css('h1.plan-name')

        if pl_number_element:
            # שולף את הטקסט של האלמנט
            text = pl_number_element.text
            # נחתוך את המידע אחרי "תוכנית" כדי לשמור את החלק השני של הטקסט
            plan_number = text.split(' ')[1]
            
            plan_attributes['attributes']['pl_number'] = plan_number
        else:
            print("pl_number: לא נמצא אלמנט עם שם המחלקה הזה!")

        pl_number_element = ''

        # מחפש את האלמנט <h2> עם מחלקה 'uk-margin-remove'
        pl_name_element = browser.find_by_css('h2.uk-margin-remove')

        if pl_name_element:
            # שולף את הטקסט של האלמנט
            text = pl_name_element.text

            plan_attributes['attributes']['pl_name'] = text
        else:
            print("pl_name: לא נמצא אלמנט עם המחלקה הזו!")

        pl_name_element = ''

        current_url = browser.url
        plan_attributes['attributes']['pl_url'] = current_url

        current_url = ''

        pl_station_desc_element = browser.find_by_css("div[role='heading'][class*='h3 uk-margin-remove']")

        if pl_station_desc_element:
            plan_attributes['attributes']['station_desc'] = pl_station_desc_element[0].text
        else:
            print("station_desc: לא נמצא אלמנט עם המחלקה הזו!")

        pl_station_desc_element = ''

        location_list_elements = browser.find_by_css("li[role='presentation'][sv4-location-list][class*='sv4-icon-flag ng-star-inserted']")

        if location_list_elements:
            sleep(2)
            location_list_elements[0].click()

            location_text_lead_elements = location_list_elements.find_by_css("div[class*='uk-width-1-2']")
            plan_attributes['attributes']['plan_county_name'] = location_text_lead_elements[2].text
        else:
            print("plan_county_name: לא נמצא אלמנט עם המחלקה הזו!")


        location_list_elements = ''
        
        # הדפסת כותרות התוצאות או מידע אחר
        buttons = browser.find_by_tag('button')
        for button in buttons:
            if button.text == 'נתונים נוספים':
                button.click()

        soup = BeautifulSoup(browser.html, 'html.parser')
        mydivs = soup.find_all("li", {"class": "sv4-icon-arrow uk-open uk-hide-arrow ng-star-inserted"})

        for divs in mydivs:
            div = divs.find_all('div', attrs={'class': 'uk-accordion-content uk-margin-remove'})

        results = div[0].find_all('ul')

        for result in results:
            li = result.find_all('li')
            lis_elements.append(li)
            
        obj = {}
        for element in lis_elements:
            quantitative_data_main_header = element[0].find_all('div', {'class': 'uk-width-expand ng-star-inserted'})[0].text.strip()
            quantitative_data_main_value = element[0].find_all('div', {'class': 'uk-width-1-2 uk-text-left'})[0].find_all('b')[0].text
            obj[quantitative_data_main_header] = quantitative_data_main_value

        plan_attributes['attributes'].update(obj)

        catA_elements = browser.find_by_css("li[role='presentation'][sv4-doc-cat-a][class*='sv4-icon-doc-purple catA']")

        if catA_elements:
            for catA_element in catA_elements:
                if catA_element.text == 'מסמכי מידע מנהלי':
                    if catA_element.visible:
                        catA_element.click()
                    else:
                        print("האלמנט אינו נראה")


        catC_elements = browser.find_by_css("li[role='presentation'][sv4-doc-cat-c][class*='catC']")

        if catC_elements:
            for element in catC_elements:
                clean_text = element.text.replace('\n', ' ').strip()  # מחליף מעברי שורה ברווחים ומנקה רווחים
                if 'קבצים דיגיטלים' in clean_text:
                    element.click()

        download_elements = browser.find_by_css("li[role='presentation'][class*='oppul']")

        if download_elements:
            for element in download_elements:
                clean_text = element.text.replace('\n', ' ').strip()  # מחליף מעברי שורה ברווחים ומנקה רווחים
                if 'קבצי התכנית (SHP)' in clean_text:
                    links_inside = element.find_by_tag('a')
                    for link in links_inside:
                        link.click()

                        sleep(7)
                    # pr# נתיב לתיקיית ההורדות
                    download_dir = r'C:\Users\dpere\Downloads'

                    # ה-ID של הקובץ שאתה מחפש
                    id_to_find = "102-0074732"

                    # רשום את כל הקבצים בתיקיית ההורדות
                    files = os.listdir(download_dir)

                    # סנן את הקבצים לפי ה-ID שמופיע בשמם
                    matching_files = [f for f in files if id_to_find in f]

                    # הדפס את כל הקבצים שמתאימים ל-ID
                    if matching_files:
                        zip_file_path = os.path.join(download_dir, matching_files[0])

                        extract_to_folder = '../ags-iplan-data-fetcher-python/background_files'
                        # extract_to_folder = r'C:\Users\dpere\Documents\JTMT\ags-iplan-data-fetcher-python\background_files'

                        # יצירת תיקיית יעד עם שם ה-ID
                        extract_to_folder = os.path.join(extract_to_folder, id_to_find)

                        # אם תיקיית היעד לא קיימת, ניצור אותה
                        os.makedirs(extract_to_folder, exist_ok=True)

                        # לפתוח את קובץ ה-ZIP
                        with zipfile.ZipFile(zip_file_path, 'r') as zip_ref:
                            # רשימה של כל הקבצים ב-ZIP
                            file_names = zip_ref.namelist()
                            
                            # סינון קבצים שמתחילים ב-'MVT_GVUL'
                            mvt_gvul_files = [f for f in file_names if f.startswith('MVT_GVUL')]
                            
                            # פרוק כל הקבצים שמתחילים ב-'MVT_GVUL'
                            for file_name in mvt_gvul_files:
                                zip_ref.extract(file_name, extract_to_folder)
                                print(f'קובץ מתוך ה-ZIP: {file_name} פורסם בנתיב: {extract_to_folder}')


                        # הדפסת שמות הקבצים המפורקים
                        for file_name in zip_ref.namelist():
                            print(f'קובץ מתוך ה-ZIP: {file_name}')
                    else:
                        print(f"לא נמצא קובץ עם ה-ID {id_to_find} להורדה.")

        pl_station_desc_element = browser.find_by_css("div[role='heading'][class*='h3 uk-margin-remove']")

        if pl_station_desc_element:
            pl_station_desc_element[0].text

        # הגדרת תיקיית ההורדות ונתיב לתיקיה חדשה עם שם ה-ID
        download_dir = r"C:\Users\dpere\Documents\JTMT\ags-iplan-data-fetcher-python\background_files"
        id_to_find = "102-0074732"  # ID שלך (שיש לקחת לפי המקרה שלך)

        # יצירת תיקיה בשם ה-ID
        folder_name = os.path.join(download_dir, id_to_find)
        # os.makedirs(folder_name, exist_ok=True)

        # הנתיב לקובץ ה-SHP (שכבה גיאוגרפית)
        shp_file = os.path.join(folder_name, 'MVT_GVUL.shp')

        # טעינת קובץ ה-SHP בעזרת GeoPandas
        gdf = gpd.read_file(shp_file)

        # כעת, נקבל את המידע הגיאומטרי - נגיד שהשכבה היא פוליגון (polygon)
        geometry = gdf.geometry.iloc[0]  # או לפי אינדקס מתאים

        # המרת הגיאומטריה למבנה של 'rings'
        rings = [list(geometry.exterior.coords)]  # מייצגים את הקואורדינטות כ-rings

        # יצירת האובייקט
        geo_object = {
            "attributes": plan_attributes['attributes'],
            "geometry": {
                "rings": rings,
            },
        }

        xplan_plans_data.append(geo_object)
    else:
        print("לא נמצא שדה חיפוש באתר!")
    
    search_box = ''

    browser.visit('https://mavat.iplan.gov.il/SV1')

In [ ]:
file_date=pd.Timestamp.today().strftime('%y%m%d')

In [ ]:
features = []

key_mapping = {
    'חדרי מלון / תיירות (חדר)': 'hotel_rooms_room',
    'חדרי מלון / תיירות (מ"ר)': 'hotel_rooms_square_meters',
    'מבני ציבור (מ"ר)': 'public_buildings_square_meters',
    'מגורים (יח"ד)': 'residence_housing_units',
    'מגורים (מ"ר)': 'residence_square_meters',
    'מסחר (מ"ר)': 'trade_square_meters',
    'תעסוקה (מ"ר)': 'employment_square_meters',
    'דירות קטנות (יח"ד)': 'small_apartments_units',
    'דירות להשכרה (יח"ד)': 'apartments_for_rent_units'
}

for plan in xplan_plans_data:
        # Create a new dictionary to store the updated attributes
    new_attributes = {}
    # Loop over the attributes dictionary in each item
    for key, value in plan['attributes'].items():
        # Check if the key needs to be replaced
        if key in key_mapping:
            # Replace the key
            new_key = key_mapping[key]
            # Add the new key with the same value
            new_attributes[new_key] = value
        else:
            # If key doesn't need to be replaced, keep it as it is
            new_attributes[key] = value
    # Update the attributes with the new dictionary
    plan['attributes'] = new_attributes

    interior_rings = plan['geometry']['rings'][1:]
    exterior_ring=plan['geometry']['rings'][0]
    polygon = Polygon(exterior_ring, holes=interior_rings)
    # if polygon_input.contains(polygon.centroid).bool():
    feature = Feature(geometry=polygon, properties=plan['attributes'])
    features.append(feature)
        
crs = {
    "type": "name",
    "properties": {
        "name": "EPSG:2039"
    }
}
        
feature_collection = FeatureCollection(features)

gdf = gpd.GeoDataFrame.from_features(feature_collection, crs='EPSG:2039')
gdf.to_file(r'my_polygon/{}_polygon.shp'.format(file_date), encoding='utf-8')
gdf